# Hypothesis Testing

## Objective

Determine whether the observed difference in conversion rates between the Advertising (ad) group and the Public Service Announcement (psa) group is statistically significant.

Business Question:

Does the advertising campaign genuinely improve conversion rates, or could the observed lift be explained by random variation?

Significance Level:

α = 0.05

Primary Test:

Two-Proportion Z-Test

Secondary Validation:

Chi-Square Test of Independence

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

from scipy.stats import chi2_contingency

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
df = pd.read_csv("../data/marketing_AB.csv")

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df = df.drop(columns=["unnamed:_0"])

In [3]:
summary = (
    df.groupby("test_group")["converted"]
      .agg(
          users="count",
          conversions="sum"
      )
)

summary

,users,conversions
test_group,,
ad,564577,14423
psa,23524,420


In [4]:
ad_conversions = summary.loc["ad", "conversions"]
ad_users = summary.loc["ad", "users"]

psa_conversions = summary.loc["psa", "conversions"]
psa_users = summary.loc["psa", "users"]

print(ad_conversions, ad_users)
print(psa_conversions, psa_users)

14423 564577
420 23524


In [5]:
count = np.array([
    ad_conversions,
    psa_conversions
])

nobs = np.array([
    ad_users,
    psa_users
])

z_stat, p_value = proportions_ztest(
    count,
    nobs
)

print("Z Statistic:", round(z_stat,4))
print("P Value:", p_value)

Z Statistic: 7.3701
P Value: 1.7052807161559714e-13


## Two-Proportion Z-Test Results

The two-proportion z-test was conducted to compare conversion rates between the Advertising (ad) group and the PSA group.

Results:

- Z Statistic = 7.37
- P Value < 0.001

Decision:

Since the p-value is substantially lower than the significance level (α = 0.05), we reject the null hypothesis.

Conclusion:

There is strong statistical evidence that conversion rates differ between the Advertising and PSA groups.

The observed lift is unlikely to be explained by random chance alone.

In [6]:
ci_low, ci_high = confint_proportions_2indep(
    count1=ad_conversions,
    nobs1=ad_users,
    count2=psa_conversions,
    nobs2=psa_users,
    method="wald"
)

print("95% Confidence Interval")
print(f"Lower Bound: {ci_low:.5f}")
print(f"Upper Bound: {ci_high:.5f}")

95% Confidence Interval
Lower Bound: 0.00595
Upper Bound: 0.00943


## Confidence Interval Interpretation

The estimated improvement in conversion rate ranges from 0.595 to 0.943 percentage points at the 95% confidence level.

Because the entire confidence interval lies above zero, the data provides strong evidence that the advertising campaign performs better than the PSA campaign.

This result reinforces the conclusion from the two-proportion z-test.

In [7]:
contingency_table = pd.crosstab(
    df["test_group"],
    df["converted"]
)

contingency_table

converted,False,True
test_group,,
ad,550154,14423
psa,23104,420


In [8]:
chi2_stat, p_value_chi2, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-Square Statistic:", round(chi2_stat,4))
print("P Value:", p_value_chi2)

Chi-Square Statistic: 54.0058
P Value: 1.9989623063390078e-13


## Statistical Decision

Two independent statistical tests were conducted:

1. Two-Proportion Z-Test
2. Chi-Square Test of Independence

Both tests produced extremely small p-values (< 0.001).

Decision:

Reject the null hypothesis.

Conclusion:

There is strong statistical evidence that the Advertising campaign achieves a different and higher conversion rate than the PSA campaign.

In [9]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [10]:
ad_rate = ad_conversions / ad_users
psa_rate = psa_conversions / psa_users

effect_size = proportion_effectsize(
    ad_rate,
    psa_rate
)

analysis = NormalIndPower()

power = analysis.power(
    effect_size=effect_size,
    nobs1=ad_users,
    ratio=psa_users/ad_users,
    alpha=0.05
)

print("Effect Size:", round(effect_size,4))
print("Statistical Power:", round(power,4))

Effect Size: 0.053
Statistical Power: 1.0


## Statistical Power Assessment

The experiment achieved a statistical power of approximately 1.00.

This indicates that the sample size was more than sufficient to detect the observed effect.

Therefore, the likelihood of a Type II Error (failing to detect a real effect) is extremely low.

Although the observed effect size is relatively small (Cohen's h = 0.053), the large sample size provides high confidence in the statistical conclusion.

In [11]:
absolute_lift = ad_rate - psa_rate

relative_lift = (
    absolute_lift
    / psa_rate
) * 100

print(f"Absolute Lift: {absolute_lift:.4%}")
print(f"Relative Lift: {relative_lift:.2f}%")

Absolute Lift: 0.7692%
Relative Lift: 43.09%


## Practical Significance Assessment

While the statistical tests indicate a highly significant result, business decisions should also consider practical significance.

Key Findings:

- Absolute Lift ≈ 0.77 percentage points
- Relative Lift ≈ 43%

Although the absolute increase appears modest, the relative improvement is substantial.

For organizations operating at large scale, even a conversion increase of less than one percentage point can translate into significant incremental revenue.

Therefore, the observed improvement is both statistically significant and practically meaningful.